In [ ]:
from dotenv import load_dotenv
from pathlib import Path
from dotenv import dotenv_values

config = dotenv_values("/home/ernie/.env")
api_path= Path('/home/ernie/.env')

load_dotenv(dotenv_path=api_path)

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from exa_py import Exa



@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""
    exa = Exa()
    return exa.search(query, contents={"highlights": True})

In [ ]:
system_prompt = """

You are a personal chef. The user will either give you a list of ingredients they have left over in their house.  Or a shot of their fridge, you can then detect ingredients which are available.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

#response = agent.invoke(
#    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
#    config
#)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "What can i cook from these ingredients in the image?"},
    {"type": "image", "base64": img_b64, "mime_type": "image/jpg"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)